# A Typical Eval Workflow

Prompt evaluation is how you replace *"this prompt feels good"* with *"this prompt scores 8.7"*. A typical workflow is **five steps**, and you can start tiny (3 records, by hand) and scale to thousands later. Everything in this section builds on this loop.

| Step | What you do |
|------|-------------|
| 1. Draft a prompt | Write a baseline prompt template with a `{question}` hole to fill |
| 2. Create an eval dataset | Collect sample inputs that represent real production traffic |
| 3. Feed through Claude | Merge each input into the template, send to Claude, collect answers |
| 4. Feed through a grader | Score each answer (typically 1–10); average the scores |
| 5. Change the prompt & repeat | Tweak the prompt, re-run, compare scores objectively |

> This notebook documents the workflow and defines the literal building blocks (the prompt template + dataset). The **live** run — actually calling Claude and a grader — is the next few lessons (`Running the eval`, `Model based grading`, `Code based grading`).

## Step 1 — Draft a prompt

Start with a baseline you want to improve. It has a `{question}` placeholder that each dataset record will fill in.

In [1]:
def build_prompt(question):
    return f"""
Please answer the user's question:

{question}
"""


print(build_prompt("What's 2+2?"))


Please answer the user's question:

What's 2+2?



## Step 2 — Create an eval dataset

The dataset is the set of sample inputs that get interpolated into the template. In the real world this might be tens, hundreds, or thousands of records — assembled by hand *or* generated by Claude (next lesson). Here, three:

In [2]:
dataset = [
    "What's 2+2?",
    "How do I make oatmeal?",
    "How far away is the Moon?",
]

dataset

["What's 2+2?", 'How do I make oatmeal?', 'How far away is the Moon?']

## Step 3 — Feed through Claude

Each question merges into the template to become a complete prompt, which goes to Claude. The first record becomes:

```
Please answer the user's question:

What's 2+2?
```

Claude answers each: `2 + 2 = 4` for the math one, cooking steps for oatmeal, the distance for the Moon. *(We build the loop that does this for real in `Running the eval`.)*

## Step 4 — Feed through a grader

A grader looks at each **(question, answer)** pair and scores it — typically 1–10, where 10 is perfect. Averaging gives one objective number for the whole prompt.

| Question | Score |
|----------|-------|
| What's 2+2? | 10 |
| How do I make oatmeal? | 4 |
| How far away is the Moon? | 9 |

Average = (10 + 4 + 9) ÷ 3 = **7.66** — our baseline score. The grader itself can be *model-based* (ask Claude to score) or *code-based* (deterministic checks) — both are covered later in this section.

## Step 5 — Change the prompt and repeat

Now iterate. Add guidance and re-run the *whole* pipeline:

In [3]:
def build_prompt_v2(question):
    return f"""
Please answer the user's question:

{question}

Answer the question with ample detail
"""


print(build_prompt_v2("How do I make oatmeal?"))


Please answer the user's question:

How do I make oatmeal?

Answer the question with ample detail



Re-running the same dataset through the same grader with `build_prompt_v2` might lift the average from **7.66 → 8.7**. Because the dataset and grader are held constant, the score delta is attributable to the *prompt change* — not luck.

## Why this matters

- **Compare versions numerically** instead of by vibe
- **Ship the best-scoring version**, with evidence
- **Keep iterating** with confidence that changes are real improvements

The rest of this section turns each step into working code: generating datasets with Claude, running the eval loop, and the two grader styles (model-based and code-based).